# 03 · Fine-tune BGE-M3

Dữ liệu từ notebook 02 (`data/processed/finetune/merged/train.jsonl`, `val.jsonl`).

- **Giai đoạn 1** – MNRL với in-batch negatives (`NoDuplicatesDataLoader`: hai câu cùng đoạn đúng không chung batch)
- **Giai đoạn 2** – MNRL với hard negative (cần chạy bước đào ở 02)
- **Chọn checkpoint** trên **val** (Điều tách riêng, không phải benchmark)
- **Đánh giá cuối** trên **benchmark v3** (200 câu) – chỉ để báo cáo, so baseline vs fine-tune
- Sau cùng: embedding lại corpus bằng model mới vào **collection mới**

Huấn luyện dùng `labor_rag/finetune/train.py::train_model`. Mặc định **không train** (`RUN_TRAINING = False`); cần GPU (Colab/Kaggle T4 trở lên, hoặc máy có CUDA).

**VRAM:** BGE-M3 có ~568M tham số. Full fine-tune với `batch_size=16`, `max_seq_length=512` cần khoảng 16 GB+ (Colab/Kaggle T4 16 GB với fp16, hoặc A100). GPU laptop 4 GB (RTX 3050) **không đủ**; nếu bắt buộc chạy máy yếu thì giảm `batch_size` xuống 4 và `max_seq_length` 256 (chất lượng MNRL giảm vì ít in-batch negatives).

## 0. Môi trường (chỉ khi chạy trên Colab/Kaggle)

In [1]:
# !git clone <repo> PBL6 && %cd PBL6
# !pip install -q "sentence-transformers>=3" datasets python-dotenv
# Chép data/processed/lao_dong_v5/chunks.jsonl và data/processed/finetune/merged/ lên máy chạy.

In [2]:
import json
import random
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "labor_rag" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from labor_rag.config import (
    EVALUATION_DIR, FINETUNE_MERGED_DIR, FINETUNE_SOURCES, LAO_DONG_DIR,
    finetune_raw_dir, finetune_source_dir,
)

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

# Bảng màu giống notebooks/dataset_analysis.ipynb
INK, INK_SECOND, INK_MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": BASELINE,
    "axes.labelcolor": INK_SECOND, "text.color": INK, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "grid.color": GRID, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
})


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def stacked_barh(df, title, colors=CAT, figsize=(9, 4)):
    """Horizontal stacked bars, one row per index value, columns as segments."""
    ax = df.plot(kind="barh", stacked=True, color=colors[: len(df.columns)], figsize=figsize, width=0.65)
    ax.invert_yaxis()
    ax.set_title(title, loc="left", color=INK, fontsize=13, pad=10)
    ax.grid(axis="x", linewidth=0.6)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="lower right")
    plt.tight_layout()
    return ax

## 1. Cấu hình

In [3]:
import torch

RUN_BASELINE = False   # đo BGE-M3 gốc trên val + benchmark v3 (encode 1481 chunk)
RUN_TRAINING = False   # huấn luyện
RUN_FINAL_EVAL = False # đo model fine-tune trên benchmark v3

CFG = dict(
    model_name="BAAI/bge-m3",
    output_dir=ROOT / "data" / "models" / "bge-m3-legal",
    epochs=3,
    lr=2e-5,
    batch_size=16,          # MNRL hưởng lợi từ batch lớn; giảm nếu thiếu VRAM
    max_seq_length=512,     # chunk dài nhất ~1000 token; 512 cắt ~1/4 chunk dài, 1024 tốn VRAM gấp đôi
    phase="both",           # "contrastive" nếu chưa có hard negative
)
TRAIN = FINETUNE_MERGED_DIR / "train.jsonl"
VAL = FINETUNE_MERGED_DIR / "val.jsonl"
CHUNKS = LAO_DONG_DIR / "chunks.jsonl"
TEST_V3 = EVALUATION_DIR / "benchmark_questions_v3.jsonl"
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA: True | NVIDIA GeForce RTX 3050 Laptop GPU


## 2. Dữ liệu

In [4]:
train = load_jsonl(TRAIN)
val = load_jsonl(VAL)
test = load_jsonl(TEST_V3)
chunks = load_jsonl(CHUNKS)
with_neg = sum(1 for s in train if s["negatives"])
print(f"train {len(train)} | có hard negative {with_neg} | val {len(val)} | test v3 {len(test)} | corpus {len(chunks)}")
display(pd.DataFrame(Counter((s["type"], s["metadata"].get("source")) for s in train).most_common(),
                     columns=["(loại, nguồn)", "số mẫu"]))
if CFG["phase"] in ("both", "hard_neg") and with_neg == 0:
    print("Chưa có hard negative → chuyển sang phase='contrastive' (chạy RUN_MINING ở notebook 02 để có giai đoạn 2).")
    CFG["phase"] = "contrastive"

train 7118 | có hard negative 0 | val 174 | test v3 201 | corpus 1481


,"(loại, nguồn)",số mẫu
0,"(synthetic_query, phamson02)",2847
1,"(real_query, vilegal)",2713
2,"(real_query, bkai)",1359
3,"(real_query, tvpl)",133
4,"(real_query, alqac)",53
5,"(real_query, zalo)",13


Chưa có hard negative → chuyển sang phase='contrastive' (chạy RUN_MINING ở notebook 02 để có giai đoạn 2).


## 3. Bộ đánh giá

`build_ir_evaluator` (train.py) đo Accuracy@k, MRR@k, nDCG@k theo `chunk_relevance`. Val dùng khi train; test v3 chỉ dùng cuối cùng.

In [5]:
from labor_rag.finetune.train import build_ir_evaluator

val_eval = build_ir_evaluator(val, chunks)
test_eval = build_ir_evaluator(test, chunks)
test_eval.name = "benchmark_v3"


def evaluate(model_path, label):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(str(model_path), device="cuda" if torch.cuda.is_available() else "cpu")
    model.max_seq_length = CFG["max_seq_length"]
    out = {}
    for name, ev in (("val", val_eval), ("test_v3", test_eval)):
        scores = ev(model)
        for k, val_ in scores.items():
            m = re.search(r"(accuracy@(?:1|5|10)|mrr@10|ndcg@10)$", k)
            if m and "cosine" in k:
                out[f"{name}/{m.group(1)}"] = round(val_, 4)
    return pd.Series(out, name=label)

results = []
if RUN_BASELINE:
    results.append(evaluate(CFG["model_name"], "bge-m3 gốc"))
    display(pd.concat(results, axis=1))

D:\Work\Learn\HK7_2026-2027\PBL6\labor_rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Huấn luyện

In [6]:
if RUN_TRAINING:
    from labor_rag.finetune.train import train_model
    train_model(
        model_name=CFG["model_name"],
        output_dir=Path(CFG["output_dir"]),
        train_path=TRAIN,
        val_path=VAL,
        chunks_path=CHUNKS,
        epochs=CFG["epochs"],
        lr=CFG["lr"],
        batch_size=CFG["batch_size"],
        max_seq_length=CFG["max_seq_length"],
        phase=CFG["phase"],
    )
else:
    print("RUN_TRAINING = False — bỏ qua huấn luyện.")

RUN_TRAINING = False — bỏ qua huấn luyện.


## 5. Đánh giá cuối trên benchmark v3

In [7]:
if RUN_FINAL_EVAL:
    results.append(evaluate(CFG["output_dir"], "bge-m3 fine-tune"))
    table = pd.concat(results, axis=1)
    if table.shape[1] == 2:
        table["chênh"] = table.iloc[:, 1] - table.iloc[:, 0]
    display(table)

## 6. Đưa model vào hệ thống

Vector trong Qdrant hiện được tạo bằng BGE-M3 gốc, nên **phải embedding lại toàn bộ corpus** bằng model fine-tune, vào **collection mới** (giữ collection cũ làm baseline):

```powershell
python -m labor_rag.ingest.embed_and_index --model data/models/bge-m3-legal --collection legal_ld_v5_bge_ft --recreate
```

Sau đó đặt `QDRANT_COLLECTION=legal_ld_v5_bge_ft` và chạy lại benchmark retrieval (`labor_rag.evaluation.embedding_benchmark`) để so trên toàn pipeline (hybrid + graph).